[![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/kgrid-objects/FAIR-DO-Workshop/HEAD?urlpath=lab/tree/HBOT3-KA/auxiliary/aux-notebook/hbot_treatment_target_ka_binder.ipynb)

## Simplified In-Notebook HBOT Treatment Target KA (adapted from src/*.js)

This section embeds a simplified reimplementation of the KA and its four constituent computations directly in the notebook, since the published `ajv`/`ajv-formats` dependency used by the real Burden KO is not installed in this Binder environment. It keeps the core Wagner scoring, HBOT Decision branching, Margolis prognosis lookup, and Burden execution-burden calculation, plus the KA's own gate mapping and twelve-cell synthesis matrix, while omitting schema validation, provenance, fingerprinting, and audit-only machinery.

Run the JavaScript cells in order.

### Wagner (copied from collection/DFU-Severity-Score-KO/src/scorer.js)

In [ ]:
const CANONICAL_QUESTION_IDS = ['Q01','Q02','Q03','Q04','Q05','Q06','Q07','Q08','Q09','Q10'];
const ANALYSIS_STATUS = {
  GRADE_COMPUTED: 'grade_computed',
  INDETERMINATE: 'indeterminate',
  AMBIGUOUS: 'ambiguous',
  INCOMPLETE_RESPONSE: 'incomplete_response',
  CONFLICTING_RESPONSE: 'conflicting_response'
};
const GRADE_LABEL_BY_SCORE = {
  0: 'At-risk foot without an open ulcer',
  1: 'Superficial ulcer',
  2: 'Deep ulcer',
  3: 'Deep ulcer with abscess or osteomyelitis',
  4: 'Localized gangrene',
  5: 'Extensive gangrene'
};

function analyzeQuestionnaireResponse(input) {
  const normalized = normalizeInput(input);
  const { question_ids, responses } = normalized;
  if (responses.includes('i')) return makeResult(question_ids, responses, ANALYSIS_STATUS.INCOMPLETE_RESPONSE);
  if (responses.includes('X')) return analyzeWithXCompletion(question_ids, responses);
  return analyzeBooleanRow(question_ids, responses);
}

function normalizeInput(input) {
  if (!input || typeof input !== 'object') throw new TypeError('Input must be an object with question_ids and responses.');
  const questionIds = input.question_ids;
  const responses = input.responses;
  if (!Array.isArray(questionIds) || !Array.isArray(responses)) throw new TypeError('question_ids and responses must both be arrays.');
  if (questionIds.length !== CANONICAL_QUESTION_IDS.length) throw new TypeError('question_ids must contain exactly 10 items in canonical order.');
  if (responses.length !== CANONICAL_QUESTION_IDS.length) throw new TypeError('responses must contain exactly 10 items aligned to question_ids.');
  for (let i = 0; i < CANONICAL_QUESTION_IDS.length; i += 1) {
    if (questionIds[i] !== CANONICAL_QUESTION_IDS[i]) throw new TypeError('question_ids must be canonical Q01-through-Q10 order.');
  }
  const normalizedResponses = responses.map((value) => normalizeResponseValue(value));
  return { question_ids: [...CANONICAL_QUESTION_IDS], responses: normalizedResponses };
}

function normalizeResponseValue(value) {
  if (value === 0 || value === '0') return '0';
  if (value === 1 || value === '1') return '1';
  if (value === 'X') return 'X';
  if (value === 'i') return 'i';
  throw new TypeError('responses values must be one of "0", "1", "X", or "i".');
}

function analyzeWithXCompletion(questionIds, responsesWithX) {
  const xIndexes = [];
  for (let i = 0; i < responsesWithX.length; i += 1) if (responsesWithX[i] === 'X') xIndexes.push(i);
  const completionCount = 2 ** xIndexes.length;
  const validOutcomes = [];
  for (let mask = 0; mask < completionCount; mask += 1) {
    const completion = [...responsesWithX];
    for (let bit = 0; bit < xIndexes.length; bit += 1) {
      const idx = xIndexes[bit];
      completion[idx] = (mask & (1 << bit)) === 0 ? '0' : '1';
    }
    const outcome = classifyBooleanRow(completion);
    if (outcome.analysis_status !== ANALYSIS_STATUS.CONFLICTING_RESPONSE) validOutcomes.push(outcome);
  }
  if (validOutcomes.length === 0) return makeResult(questionIds, responsesWithX, ANALYSIS_STATUS.CONFLICTING_RESPONSE);
  const allGradeComputed = validOutcomes.every((o) => o.analysis_status === ANALYSIS_STATUS.GRADE_COMPUTED);
  if (allGradeComputed) {
    const firstScore = validOutcomes[0].wagner_score[0];
    const sameScore = validOutcomes.every((o) => o.wagner_score[0] === firstScore);
    if (sameScore) return makeResult(questionIds, responsesWithX, ANALYSIS_STATUS.GRADE_COMPUTED, [firstScore], [GRADE_LABEL_BY_SCORE[firstScore]]);
  }
  const allIndeterminate = validOutcomes.every((o) => o.analysis_status === ANALYSIS_STATUS.INDETERMINATE);
  if (allIndeterminate) return makeResult(questionIds, responsesWithX, ANALYSIS_STATUS.INDETERMINATE);
  return makeResult(questionIds, responsesWithX, ANALYSIS_STATUS.CONFLICTING_RESPONSE);
}

function analyzeBooleanRow(questionIds, responses) {
  const outcome = classifyBooleanRow(responses);
  return makeResult(questionIds, responses, outcome.analysis_status, outcome.wagner_score, outcome.grade_label);
}

function classifyBooleanRow(responses) {
  const q = makeQuestionAccessor(responses);
  if (violatesHardConstraints(q)) return { analysis_status: ANALYSIS_STATUS.CONFLICTING_RESPONSE, wagner_score: [], grade_label: [] };
  const truePredicates = [];
  if (q('Q01') === 1) truePredicates.push(5);
  if (q('Q01') === 0 && q('Q02') === 1) truePredicates.push(4);
  if (q('Q01') === 0 && q('Q02') === 0 && q('Q06') === 1 && q('Q07') === 1 && (q('Q03') === 1 || q('Q04') === 1 || q('Q05') === 1)) truePredicates.push(3);
  if (q('Q01') === 0 && q('Q02') === 0 && q('Q06') === 1 && q('Q07') === 1 && q('Q03') === 0 && q('Q04') === 0 && q('Q05') === 0) truePredicates.push(2);
  if (q('Q01') === 0 && q('Q02') === 0 && q('Q06') === 1 && q('Q07') === 0 && q('Q08') === 1 && q('Q03') === 0 && q('Q04') === 0 && q('Q05') === 0) truePredicates.push(1);
  if (q('Q01') === 0 && q('Q02') === 0 && q('Q06') === 0 && q('Q10') === 1) truePredicates.push(0);
  if (truePredicates.length === 1) {
    const score = truePredicates[0];
    return { analysis_status: ANALYSIS_STATUS.GRADE_COMPUTED, wagner_score: [score], grade_label: [GRADE_LABEL_BY_SCORE[score]] };
  }
  if (truePredicates.length > 1) return { analysis_status: ANALYSIS_STATUS.AMBIGUOUS, wagner_score: [], grade_label: [] };
  return { analysis_status: ANALYSIS_STATUS.INDETERMINATE, wagner_score: [], grade_label: [] };
}

function violatesHardConstraints(q) {
  if (q('Q01') === 1 && q('Q02') === 1) return true;
  if (q('Q01') === 1 && (q('Q06') !== 1 || q('Q09') !== 0 || q('Q10') !== 0)) return true;
  if (q('Q02') === 1 && (q('Q06') !== 1 || q('Q09') !== 0 || q('Q10') !== 0)) return true;
  if (q('Q06') === 1 && (q('Q09') !== 0 || q('Q10') !== 0)) return true;
  if (q('Q10') === 1 && (q('Q06') !== 0 || q('Q09') !== 1)) return true;
  if (q('Q10') === 1 && (q('Q03') === 1 || q('Q04') === 1 || q('Q05') === 1)) return true;
  if (q('Q06') === 0 && (q('Q07') !== 0 || q('Q08') !== 0)) return true;
  if (q('Q07') === 1 && (q('Q06') !== 1 || q('Q08') !== 0)) return true;
  if (q('Q08') === 1 && (q('Q06') !== 1 || q('Q07') !== 0)) return true;
  return false;
}

function makeQuestionAccessor(responses) {
  const map = {};
  for (let i = 0; i < CANONICAL_QUESTION_IDS.length; i += 1) map[CANONICAL_QUESTION_IDS[i]] = responses[i] === '1' ? 1 : 0;
  return (id) => map[id];
}

function makeResult(questionIds, responses, status, score = [], label = []) {
  return { question_ids: questionIds, responses, analysis_status: status, wagner_score: score, grade_label: label };
}

console.log('Wagner module loaded.');

### HBOT Decision (simplified from collection/HBOT-Treatment-Decision-KO/src/decision.js)

Keeps the core branch order (dfu_confirmed, wagner_grade, acute_surgical_intervention, not_healed_after_30_days) while omitting the serialized-input parser, duplicate-property detection, and fixture adapter.

In [ ]:
const HBOT_INPUT_KEYS = {
  IN_01: 'dfu_confirmed',
  IN_02: 'wagner_grade',
  IN_03: 'acute_surgical_intervention',
  IN_04: 'not_healed_after_30_days'
};

function hbotEvaluateInputObject(inputObject) {
  if (inputObject === null || Array.isArray(inputObject) || typeof inputObject !== 'object') {
    throw new TypeError('Top-level input must be a JSON object.');
  }

  const in01 = inputObject[HBOT_INPUT_KEYS.IN_01];
  const in02 = inputObject[HBOT_INPUT_KEYS.IN_02];
  const in03 = inputObject[HBOT_INPUT_KEYS.IN_03];
  const in04 = inputObject[HBOT_INPUT_KEYS.IN_04];

  if (typeof in01 !== 'string' || (in01 !== 'true' && in01 !== 'false')) {
    return { status: 'error', result_id: 'ERR-09', error_field: 'IN-01' };
  }
  if (!Number.isInteger(in02) || in02 < 0 || in02 > 5) {
    return { status: 'error', result_id: 'ERR-02', error_field: 'IN-02' };
  }
  if (typeof in03 !== 'string' || (in03 !== 'true' && in03 !== 'false')) {
    return { status: 'error', result_id: 'ERR-09', error_field: 'IN-03' };
  }
  if (typeof in04 !== 'string' || (in04 !== 'true' && in04 !== 'false')) {
    return { status: 'error', result_id: 'ERR-09', error_field: 'IN-04' };
  }

  if (in01 === 'false') return { status: 'out-of-scope', result_id: 'OUT-OF-SCOPE' };
  if (in02 <= 2) return { status: 'completed', result_id: 'OUTPUT-01' };
  if (in03 === 'true') return { status: 'completed', result_id: 'OUTPUT-02' };
  if (in04 === 'true') return { status: 'completed', result_id: 'OUTPUT-03' };
  return { status: 'completed', result_id: 'OUTPUT-04' };
}

console.log('HBOT Decision module loaded.');

### Margolis (copied from collection/DFU-Prognostic-Indicator-KO/src/prognosis.js)

Keeps the core area/duration rule lookup while omitting input validation diagnostics and the rule-cardinality fault-injection variant.

In [ ]:
const MARGOLIS_RULES = {
  'AREA-GE-2|DUR-GE-8': ['ALG-01', 'AREA_GE_2__DURATION_GE_8', 0.203, '20.3%'],
  'AREA-GE-2|DUR-LT-8': ['ALG-02', 'AREA_GE_2__DURATION_LT_8', 0.304, '30.4%'],
  'AREA-LT-2|DUR-GE-8': ['ALG-03', 'AREA_LT_2__DURATION_GE_8', 0.448, '44.8%'],
  'AREA-LT-2|DUR-LT-8': ['ALG-04', 'AREA_LT_2__DURATION_LT_8', 0.631, '63.1%']
};

function margolisEvaluate(input) {
  const area = input.wound_area;
  const duration = input.wound_duration;
  const areaValue = area.ucum_code === 'cm2' ? area.value : area.value / 100;
  const durationValue = duration.ucum_code === 'wk' ? duration.value : duration.value / 7;
  const areaCategory = areaValue >= 2 ? 'AREA-GE-2' : 'AREA-LT-2';
  const durationCategory = durationValue >= 8 ? 'DUR-GE-8' : 'DUR-LT-8';
  const rule = MARGOLIS_RULES[`${areaCategory}|${durationCategory}`];
  return {
    status: 'success',
    result_id: rule[0],
    prognostic_group: rule[1],
    healing_probability_16_weeks: rule[2],
    display_probability_percent: rule[3]
  };
}

console.log('Margolis module loaded.');

### Burden (simplified, adapted from collection/HBOT-Regimen-Burden-KO's own simplified in-notebook example)

Keeps the core travel-band/attendance execution-burden calculation while omitting schema validation, the provider roster, provenance, fingerprinting, and audit-only machinery. `weekday_attendance_difficulty` and travel time/miles are provided directly instead of through the interactive questionnaire runner.

In [ ]:
function classifyTravelBand(oneWayMinutes) {
  const roundTripMinutes = oneWayMinutes * 2;
  if (roundTripMinutes < 60) return 'low';
  if (roundTripMinutes < 120) return 'moderate';
  if (roundTripMinutes < 180) return 'high';
  return 'very_high';
}

function burdenCalculate(answers) {
  const travelBand = classifyTravelBand(answers.one_way_travel_minutes);
  const travelScore = { low: 1, moderate: 2, high: 3, very_high: 4 }[travelBand];
  const difficulty = answers.weekday_attendance_difficulty;

  let score;
  if (difficulty === 'none') score = travelScore;
  else if (difficulty === 'some') score = Math.min(5, travelScore + 1);
  else score = travelScore >= 3 ? 5 : 4;

  const categories = [
    'low_execution_burden',
    'moderate_execution_burden',
    'high_execution_burden',
    'very_high_execution_burden',
    'extreme_execution_burden'
  ];
  const category = categories[score - 1];

  return {
    status: 'completed',
    result_code: `RESULT-${category.toUpperCase().replace(/_/g, '-')}`,
    execution_burden_level: { score, category }
  };
}

console.log('Burden module loaded.');

### KA-owned gate mapping and synthesis (Sections 6.2-6.4, 7.8-7.10, copied from src/gate-mapping.js, src/synthesis.js, and src/orchestrator.js)

This is the only logic genuinely owned by the KA itself: mapping the HBOT Decision native result to a gate state, projecting the native Margolis/Burden results onto approved bands, and looking up the twelve-cell synthesis matrix. It never recomputes Wagner grade, HBOT branch order, burden score, or prognosis -- it only reads the already-computed native results produced above.

In [ ]:
const GATE_BY_NATIVE_RESULT = {
  'OUTPUT-01': 'NOT_SUPPORTED',
  'OUTPUT-02': 'SUPPORTED',
  'OUTPUT-03': 'SUPPORTED',
  'OUTPUT-04': 'INSUFFICIENT_DECISION',
  'OUT-OF-SCOPE': 'OUT_OF_SCOPE'
};

const PROGNOSIS_PROJECTION = {
  AREA_GE_2__DURATION_GE_8: 'VERY_UNFAVORABLE_PROGNOSIS_BAND',
  AREA_GE_2__DURATION_LT_8: 'UNFAVORABLE_PROGNOSIS_BAND',
  AREA_LT_2__DURATION_GE_8: 'INTERMEDIATE_PROGNOSIS_BAND',
  AREA_LT_2__DURATION_LT_8: 'RELATIVELY_FAVORABLE_PROGNOSIS_BAND'
};

const BURDEN_PROJECTION = {
  low_execution_burden: 'LOWER_BURDEN_BAND',
  moderate_execution_burden: 'MIDDLE_BURDEN_BAND',
  high_execution_burden: 'HIGHER_BURDEN_BAND',
  very_high_execution_burden: 'HIGHER_BURDEN_BAND',
  extreme_execution_burden: 'HIGHER_BURDEN_BAND'
};

const SYNTHESIS_MATRIX = {
  'VERY_UNFAVORABLE_PROGNOSIS_BAND|LOWER_BURDEN_BAND': ['SYN-VU-L-01', 'ON_TARGET'],
  'VERY_UNFAVORABLE_PROGNOSIS_BAND|MIDDLE_BURDEN_BAND': ['SYN-VU-M-02', 'ON_TARGET'],
  'VERY_UNFAVORABLE_PROGNOSIS_BAND|HIGHER_BURDEN_BAND': ['SYN-VU-H-03', 'NEAR_TARGET'],
  'UNFAVORABLE_PROGNOSIS_BAND|LOWER_BURDEN_BAND': ['SYN-U-L-04', 'ON_TARGET'],
  'UNFAVORABLE_PROGNOSIS_BAND|MIDDLE_BURDEN_BAND': ['SYN-U-M-05', 'NEAR_TARGET'],
  'UNFAVORABLE_PROGNOSIS_BAND|HIGHER_BURDEN_BAND': ['SYN-U-H-06', 'NEAR_TARGET'],
  'INTERMEDIATE_PROGNOSIS_BAND|LOWER_BURDEN_BAND': ['SYN-I-L-07', 'NEAR_TARGET'],
  'INTERMEDIATE_PROGNOSIS_BAND|MIDDLE_BURDEN_BAND': ['SYN-I-M-08', 'NEAR_TARGET'],
  'INTERMEDIATE_PROGNOSIS_BAND|HIGHER_BURDEN_BAND': ['SYN-I-H-09', 'OUTER_TARGET'],
  'RELATIVELY_FAVORABLE_PROGNOSIS_BAND|LOWER_BURDEN_BAND': ['SYN-RF-L-10', 'NEAR_TARGET'],
  'RELATIVELY_FAVORABLE_PROGNOSIS_BAND|MIDDLE_BURDEN_BAND': ['SYN-RF-M-11', 'OUTER_TARGET'],
  'RELATIVELY_FAVORABLE_PROGNOSIS_BAND|HIGHER_BURDEN_BAND': ['SYN-RF-H-12', 'OUTER_TARGET']
};

function runHbotTreatmentTargetKA(kaInput) {
  const wagnerNative = analyzeQuestionnaireResponse(kaInput.wagner_response);
  if (wagnerNative.analysis_status !== 'grade_computed' || wagnerNative.wagner_score.length !== 1) {
    return { status: 'indeterminate', target_classification: 'INDETERMINATE', reason_code: 'KA-ERR-CONSTITUENT-RESULT', native_results: { wagner: wagnerNative } };
  }

  const wagnerGrade = wagnerNative.wagner_score[0]; // TX-01
  const assertions = kaInput.hbot_case_assertions;
  const hbotRequest = {
    dfu_confirmed: assertions.dfu_confirmed ? 'true' : 'false', // TX-02
    wagner_grade: wagnerGrade,
    acute_surgical_intervention: assertions.acute_surgical_intervention ? 'true' : 'false',
    not_healed_after_30_days: assertions.not_healed_after_30_days ? 'true' : 'false'
  };
  const hbotNative = hbotEvaluateInputObject(hbotRequest);
  const gateResult = GATE_BY_NATIVE_RESULT[hbotNative.result_id] || null;

  if (gateResult === null) {
    return { status: 'indeterminate', target_classification: 'INDETERMINATE', reason_code: 'KA-ERR-GATE-UNDEFINED', gate_result: gateResult, native_results: { wagner: wagnerNative, hbot_decision: hbotNative } };
  }
  if (gateResult === 'NOT_SUPPORTED') {
    return { status: 'completed', target_classification: 'OFF_TARGET', reason_code: 'KA-RESULT-HBOT-NOT-SUPPORTED', gate_result: gateResult, native_results: { wagner: wagnerNative, hbot_decision: hbotNative } };
  }
  if (gateResult === 'INSUFFICIENT_DECISION') {
    return { status: 'indeterminate', target_classification: 'INDETERMINATE', reason_code: 'KA-INDET-HBOT-INSUFFICIENT-DECISION', gate_result: gateResult, native_results: { wagner: wagnerNative, hbot_decision: hbotNative } };
  }
  if (gateResult === 'OUT_OF_SCOPE') {
    return { status: 'indeterminate', target_classification: 'INDETERMINATE', reason_code: 'KA-INDET-HBOT-OUT-OF-SCOPE', gate_result: gateResult, native_results: { wagner: wagnerNative, hbot_decision: hbotNative } };
  }

  // gateResult === 'SUPPORTED'
  const margolisNative = margolisEvaluate(kaInput.margolis_first_visit_assessment);
  const burdenNative = burdenCalculate(kaInput.burden_questionnaire_answers);
  const prognosisBand = PROGNOSIS_PROJECTION[margolisNative.prognostic_group] || null;
  const burdenBand = BURDEN_PROJECTION[burdenNative.execution_burden_level.category] || null;

  if (!prognosisBand || !burdenBand) {
    return { status: 'indeterminate', target_classification: 'INDETERMINATE', reason_code: 'KA-ERR-SYNTHESIS-UNKNOWN-PROJECTION', gate_result: gateResult, native_results: { wagner: wagnerNative, hbot_decision: hbotNative, margolis: margolisNative, burden: burdenNative } };
  }

  const hit = SYNTHESIS_MATRIX[`${prognosisBand}|${burdenBand}`];
  if (!hit) {
    return { status: 'indeterminate', target_classification: 'INDETERMINATE', reason_code: 'KA-ERR-SYNTHESIS-COMBINATION-UNDEFINED', gate_result: gateResult, native_results: { wagner: wagnerNative, hbot_decision: hbotNative, margolis: margolisNative, burden: burdenNative } };
  }

  return {
    status: 'completed',
    target_classification: hit[1],
    reason_code: 'KA-RESULT-SUPPORTED-SYNTHESIS',
    gate_result: gateResult,
    synthesis_rule_id: hit[0],
    prognosis_band: prognosisBand,
    burden_band: burdenBand,
    native_results: { wagner: wagnerNative, hbot_decision: hbotNative, margolis: margolisNative, burden: burdenNative }
  };
}

console.log('HBOT Treatment Target KA orchestration loaded.');

### Demo

This target-zone classification describes position within this knowledge assembly only. It does not predict HBOT efficacy or benefit, recommend or authorize treatment, rank or prioritize patients, or calculate treatment utility.

In [ ]:
const sampleInput = {
  wagner_response: {
    question_ids: ['Q01','Q02','Q03','Q04','Q05','Q06','Q07','Q08','Q09','Q10'],
    responses: ['0', '0', '1', '0', '0', '1', '1', '0', '0', '0']
  },
  hbot_case_assertions: {
    dfu_confirmed: true,
    acute_surgical_intervention: true,
    not_healed_after_30_days: false
  },
  margolis_first_visit_assessment: {
    wound_area: { value: 1, ucum_code: 'cm2' },
    wound_duration: { value: 4, ucum_code: 'wk' }
  },
  burden_questionnaire_answers: {
    hyperbaric_oxygen_therapy_location: 'https://kgrid.org/cks/dfu-hbot-burden/providers/e-001',
    one_way_miles: 5,
    one_way_travel_minutes: 10,
    weekday_attendance_difficulty: 'none'
  }
};

const result = runHbotTreatmentTargetKA(sampleInput);
console.log(JSON.stringify(result, null, 2));